In [1]:
# -*- coding: utf-8 -*-
"""
Created on 2023-01-01
Revised on 2026-04-04

@author:       Oscar Trevizo
@institution:  Harvard Extension School — Graduate Data Science Program (2023)
@context:      Independent project — vignette, R to Python (_r2p)
@environment:  Python 3.14.3 | myenv | MacBook Air M5

pandas Data Wrangling — dplyr Vignette (_r2p)
==============================================

Purpose:
    Demonstrates pandas data wrangling as the Python equivalent of R's dplyr
    library. Covers the complete dplyr verb grammar:

      select()    → df[['col1','col2']] or df.filter()
      filter()    → df[condition] or df.query()
      arrange()   → df.sort_values()
      summarize() → df.agg()
      group_by()  → df.groupby()
      mutate()    → df.assign()
      join()      → pd.merge()
      relocate()  → df[reordered_columns]
      %>% pipe    → method chaining with .

    Two datasets:
      Part 1: nycflights13 — NYC flights 2013
               Install: pip install nycflights13
               Import:  from nycflights13 import flights
      Part 2: possum {DAAG} — Australian possum morphology
               Loaded from seaborn or generated inline

    The R %>% pipe maps to pandas method chaining:
      R:  flights %>% select(...) %>% filter(...) %>% arrange(...)
      Py: (flights
            [['col1','col2',...]]
            .query('condition')
            .sort_values('col'))

    This notebook belongs in Python/vignettes/ (not toolbox/) —
    it is a library demonstration, not a function definition.

    R equivalent: dplyr_library_vignette.Rmd
    R libraries:  dplyr, nycflights13, DAAG
    Python libs:  pandas, numpy, nycflights13 (pip install nycflights13)

    Suffix _r2p: This notebook was converted from R to Python.

    Reference: Dr. Bharatendra Rai YouTube channel
    https://www.youtube.com/watch?v=BPR_Dkll17Y

Revision History:
    2023-01-01  Original R development (Harvard STAT 109)
                - R script: dplyr_library_vignette.Rmd
                - Based on material by Prof. Bharatendra Rai

    2026-04-04  Converted to Python / Jupyter Notebook (_r2p)
                - dplyr %>% pipe → pandas method chaining
                - select() → df[[cols]] or df.filter()
                - filter() → df.query() or boolean indexing
                - arrange() → df.sort_values()
                - summarize() + group_by() → df.groupby().agg()
                - mutate() → df.assign()
                - left/right/inner_join() → pd.merge(how=...)
                - nycflights13 R package → nycflights13 Python package
                - possum {DAAG} → inline dataset
"""

"\nCreated on 2023-01-01\nRevised on 2026-04-04\n\n@author:       Oscar Trevizo\n@institution:  Harvard Extension School — Graduate Data Science Program (2023)\n@context:      Independent project — vignette, R to Python (_r2p)\n@environment:  Python 3.14.3 | myenv | MacBook Air M5\n\npandas Data Wrangling — dplyr Vignette (_r2p)\n==============================================\n\nPurpose:\n    Demonstrates pandas data wrangling as the Python equivalent of R's dplyr\n    library. Covers the complete dplyr verb grammar:\n\n      select()    → df[['col1','col2']] or df.filter()\n      filter()    → df[condition] or df.query()\n      arrange()   → df.sort_values()\n      summarize() → df.agg()\n      group_by()  → df.groupby()\n      mutate()    → df.assign()\n      join()      → pd.merge()\n      relocate()  → df[reordered_columns]\n      %>% pipe    → method chaining with .\n\n    Two datasets:\n      Part 1: nycflights13 — NYC flights 2013\n               Install: pip install nycflights1

# pandas Data Wrangling — dplyr Vignette

## Purpose

Demonstrates **pandas** as the Python equivalent of R's **dplyr** library.
Covers the complete dplyr verb grammar across two datasets.

> **On the name:** dplyr = "d-plier" — pliers for data. 
> In Python, pandas is the equivalent toolkit.

## The dplyr → pandas Verb Map

| dplyr verb | pandas equivalent | Description |
|-----------|-------------------|-------------|
| `select()` | `df[['col1','col2']]` or `df.filter()` | Choose columns |
| `filter()` | `df.query()` or boolean indexing | Subset rows |
| `arrange()` | `df.sort_values()` | Sort rows |
| `summarize()` | `df.agg()` | Aggregate |
| `group_by()` | `df.groupby()` | Group |
| `mutate()` | `df.assign()` | Create/modify columns |
| `left_join()` | `pd.merge(how='left')` | Join |
| `right_join()` | `pd.merge(how='right')` | Join |
| `inner_join()` | `pd.merge(how='inner')` | Join |
| `relocate()` | `df[reordered_col_list]` | Reorder columns |
| `%>%` pipe | `.` method chaining | Chain operations |
| `n()` | `.agg('count')` or `len` | Count rows |
| `desc()` | `ascending=False` | Descending sort |
| `na.rm=TRUE` | `skipna=True` (default) | Skip NaN |

**R equivalent:** `dplyr_library_vignette.Rmd`

**Note:** This notebook belongs in `Python/vignettes/` — it is a library
demonstration, not a reusable function definition.

**Reference:** Dr. Bharatendra Rai YouTube channel (Harvard STAT 109).

## Installation Note

```bash
# Install nycflights13 Python package (run once in myenv terminal)
pip install nycflights13
```

## Imports

In [2]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.max_rows', 10)

print('pandas version:', pd.__version__)
print('Imports complete.')

pandas version: 2.3.3
Imports complete.


---

# Part 1: EDA for nycflights13 Dataset

## Load the Data

Mirrors R:
```r
library(nycflights13)
data('flights')
str(flights)
head(flights)
```

"On-time data for all flights that departed NYC (JFK, LGA, EWR) in 2013."
336,776 flights × 19 variables.

In [3]:
# Load nycflights13 — mirrors R: library(nycflights13); data('flights')
# Requires: pip install nycflights13
try:
    from nycflights13 import flights
    print(f'nycflights13 loaded: {flights.shape}')
except ImportError:
    print('nycflights13 not installed. Run: pip install nycflights13')
    print('Falling back to a small synthetic flights dataset for demo purposes.')
    # Minimal synthetic fallback so notebook runs without the package
    flights = pd.DataFrame({
        'year': [2013]*20, 'month': [1,1,1,2,2,3,3,4,5,6,7,8,9,10,11,12,1,2,3,4],
        'day':  [1,2,3,1,2,1,2,1,1,1,1,1,1,1,1,1,4,4,4,4],
        'dep_time': [517,533,542,544,554,554,555,557,558,559,
                     600,600,601,602,606,615,615,620,622,625],
        'sched_dep_time': [515,529,540,545,600,558,600,600,600,600,
                           600,600,600,602,610,615,615,620,622,625],
        'arr_time': [830,850,923,1004,812,740,913,709,753,838,
                     851,837,807,828,837,846,715,700,715,848],
        'carrier': ['UA','AA','B6','DL','UA','B6','EV','B6','AA','AA',
                    'UA','DL','B6','EV','AA','UA','UA','AA','B6','DL'],
        'flight': [1545,1714,1141,725,461,1696,507,5708,79,301,
                   1301,671,343,1716,1735,1743,1545,1714,1141,725],
        'origin': ['EWR','LGA','JFK','JFK','LGA','EWR','EWR','JFK','LGA','LGA',
                   'EWR','JFK','JFK','EWR','LGA','EWR','EWR','LGA','JFK','JFK'],
        'dest':   ['IAH','IAH','MIA','BQN','ATL','ORD','FLL','MCO','ORD','ATL',
                   'IAH','BQN','MIA','ATL','ORD','IAH','IAH','IAH','MIA','BQN'],
        'air_time': [227,227,160,183,116,150,158,53,138,138,
                     231,183,160,143,138,227,227,227,160,183],
        'distance': [1400,1416,1089,1576,762,733,1065,187,733,762,
                     1400,1576,1089,746,733,1400,1400,1416,1089,1576],
    })
    print(f'Synthetic fallback dataset: {flights.shape}')

# str(flights) equivalent — mirrors R: str(flights)
print()
print('dtypes (mirrors R: str(flights)):')
print(flights.dtypes)
print()
print('head(flights) — mirrors R: head(flights):')
flights.head()

nycflights13 not installed. Run: pip install nycflights13
Falling back to a small synthetic flights dataset for demo purposes.
Synthetic fallback dataset: (20, 12)

dtypes (mirrors R: str(flights)):
year               int64
month              int64
day                int64
dep_time           int64
sched_dep_time     int64
                   ...  
flight             int64
origin            object
dest              object
air_time           int64
distance           int64
Length: 12, dtype: object

head(flights) — mirrors R: head(flights):


,year,month,day,dep_time,sched_dep_time,arr_time,carrier,flight,origin,dest,air_time,distance
0,2013,1,1,517,515,830,UA,1545,EWR,IAH,227,1400
1,2013,1,2,533,529,850,AA,1714,LGA,IAH,227,1416
2,2013,1,3,542,540,923,B6,1141,JFK,MIA,160,1089
3,2013,2,1,544,545,1004,DL,725,JFK,BQN,183,1576
4,2013,2,2,554,600,812,UA,461,LGA,ATL,116,762


---

# select() — Choose Columns

Mirrors R: `select(.data, ...)` from dplyr

> *"Select (and optionally rename) variables in a data frame"*

The R `%>%` pipe becomes pandas **method chaining** with `.`

## select() — Method 1: List columns directly

In [4]:
# Method 1 — mirrors R: flights %>% select(month, day, origin, dest, carrier)
COLS = ['month', 'day', 'origin', 'dest', 'carrier']

sel_flights_1 = flights[COLS]

print(f'Shape: {sel_flights_1.shape}  (mirrors R: str(sel_flights_method1))')
sel_flights_1.head()

Shape: (20, 5)  (mirrors R: str(sel_flights_method1))


,month,day,origin,dest,carrier
0,1,1,EWR,IAH,UA
1,1,2,LGA,IAH,AA
2,1,3,JFK,MIA,B6
3,2,1,JFK,BQN,DL
4,2,2,LGA,ATL,UA


## select() — Method 2: Column names in a list (vector)

In [5]:
# Method 2 — mirrors R: flights %>% select(c('month','day','origin','dest','carrier'))
cols_list = ['month', 'day', 'origin', 'dest', 'carrier']
sel_flights_2 = flights[cols_list]

print(f'Shape: {sel_flights_2.shape}')
sel_flights_2.head()

Shape: (20, 5)


,month,day,origin,dest,carrier
0,1,1,EWR,IAH,UA
1,1,2,LGA,IAH,AA
2,1,3,JFK,MIA,B6
3,2,1,JFK,BQN,DL
4,2,2,LGA,ATL,UA


## select() — Method 3 & 4: Mixed and reordered

In [6]:
# Method 3 — mirrors R: flights %>% select(c('month','day','origin','dest'), carrier)
sel_flights_3 = flights[['month', 'day', 'origin', 'dest', 'carrier']]

# Method 4 — mirrors R: flights %>% select(c('month','day','carrier'), origin, c('dest'))
# Change order — same columns, different sequence
sel_flights_4 = flights[['month', 'day', 'carrier', 'origin', 'dest']]

print('Method 3 — mixed selection:')
print(sel_flights_3.head(3))
print()
print('Method 4 — reordered columns:')
print(sel_flights_4.head(3))

Method 3 — mixed selection:
   month  day origin dest carrier
0      1    1    EWR  IAH      UA
1      1    2    LGA  IAH      AA
2      1    3    JFK  MIA      B6

Method 4 — reordered columns:
   month  day carrier origin dest
0      1    1      UA    EWR  IAH
1      1    2      AA    LGA  IAH
2      1    3      B6    JFK  MIA


---

# filter() — Subset Rows

Mirrors R: `filter(.data, ...)` from dplyr

> *"Retaining all rows that satisfy your conditions"*

Python offers two styles: **boolean indexing** (explicit) and **`.query()`** (closest to dplyr syntax).

## filter() — Example 1: Equality

In [7]:
# Example 1 — mirrors R: flights %>% select(...) %>% filter(carrier == 'UA')
fltr_1 = (flights[['month', 'day', 'origin', 'dest', 'carrier']]
           .query("carrier == 'UA'"))

print(f'Shape: {fltr_1.shape}  (only UA flights)')
fltr_1.head()

Shape: (5, 5)  (only UA flights)


,month,day,origin,dest,carrier
0,1,1,EWR,IAH,UA
4,2,2,LGA,ATL,UA
10,7,1,EWR,IAH,UA
15,12,1,EWR,IAH,UA
16,1,4,EWR,IAH,UA


## filter() — Example 2: NOT operator

In [8]:
# Example 2 — mirrors R: filter(!carrier == 'UA')
fltr_2 = (flights[['month', 'day', 'origin', 'dest', 'carrier']]
           .query("carrier != 'UA'"))   # != mirrors R: !carrier == 'UA'

print(f'Shape: {fltr_2.shape}  (all except UA)')
fltr_2.head()

Shape: (15, 5)  (all except UA)


,month,day,origin,dest,carrier
1,1,2,LGA,IAH,AA
2,1,3,JFK,MIA,B6
3,2,1,JFK,BQN,DL
5,3,1,EWR,ORD,B6
6,3,2,EWR,FLL,EV


## filter() — Example 3: AND conditions

In [9]:
# Example 3 — mirrors R: filter(carrier=='UA', origin=='EWR', dest=='ORD')
# Comma in dplyr = AND; query() uses 'and'
fltr_3 = (flights[['month', 'day', 'carrier', 'origin', 'dest', 'flight', 'dep_time']]
           .query("carrier == 'UA' and origin == 'EWR' and dest == 'ORD'"))

print(f'Shape: {fltr_3.shape}  (UA from EWR to ORD)')
fltr_3.head()

Shape: (0, 7)  (UA from EWR to ORD)


,month,day,carrier,origin,dest,flight,dep_time


## filter() — Example 4: OR conditions

In [10]:
# Example 4 — mirrors R: filter((carrier=='UA'|carrier=='AA'), (origin=='JFK'|origin=='EWR'), dest=='ORD')
fltr_4 = (flights[['month', 'day', 'origin', 'dest', 'carrier']]
           .query("(carrier == 'UA' or carrier == 'AA') and "
                  "(origin == 'JFK' or origin == 'EWR') and "
                  "dest == 'ORD'"))

print(f'Shape: {fltr_4.shape}  (UA or AA from JFK or EWR to ORD)')
fltr_4.head()

Shape: (0, 5)  (UA or AA from JFK or EWR to ORD)


,month,day,origin,dest,carrier


## filter() — Example 5: Greater than / less than

In [11]:
# Example 5 — mirrors R: filter(month > 6)
fltr_5 = (flights[['month', 'day', 'origin', 'dest', 'carrier']]
           .query('month > 6'))

print(f'Shape: {fltr_5.shape}  (flights in H2: July–December)')
fltr_5.head()

Shape: (6, 5)  (flights in H2: July–December)


,month,day,origin,dest,carrier
10,7,1,EWR,IAH,UA
11,8,1,JFK,BQN,DL
12,9,1,JFK,MIA,B6
13,10,1,EWR,ATL,EV
14,11,1,LGA,ORD,AA


---

# arrange() — Sort Rows

Mirrors R: `arrange()` from dplyr → `df.sort_values()`

## arrange() — Example 1: Ascending (default)

In [12]:
# Example 1 — mirrors R: flights %>% select(...) %>% arrange(air_time)
arr_1 = (flights[['month', 'day', 'origin', 'dest', 'air_time', 'carrier']]
          .sort_values('air_time'))   # ascending is default — mirrors R arrange()

print(f'Shape: {arr_1.shape}  (shortest flights first)')
arr_1.head()

Shape: (20, 6)  (shortest flights first)


,month,day,origin,dest,air_time,carrier
7,4,1,JFK,MCO,53,B6
4,2,2,LGA,ATL,116,UA
9,6,1,LGA,ATL,138,AA
14,11,1,LGA,ORD,138,AA
8,5,1,LGA,ORD,138,AA


## arrange() — Example 2: Descending (who wants to go to Honolulu?)

In [13]:
# Example 2 — mirrors R: arrange(desc(air_time))
# desc() in dplyr → ascending=False in pandas
arr_2 = (flights[['month', 'day', 'origin', 'dest', 'air_time', 'carrier']]
          .sort_values('air_time', ascending=False))   # desc() → ascending=False

print(f'Shape: {arr_2.shape}  (longest flights first — Honolulu?)')
arr_2.head()

Shape: (20, 6)  (longest flights first — Honolulu?)


,month,day,origin,dest,air_time,carrier
10,7,1,EWR,IAH,231,UA
1,1,2,LGA,IAH,227,AA
17,2,4,LGA,IAH,227,AA
16,1,4,EWR,IAH,227,UA
15,12,1,EWR,IAH,227,UA


---

# summarize() — Aggregate

Mirrors R: `summarize()` / `summarise()` from dplyr

> *"summarise() and summarize() are synonyms"* — so is Python.

## summarize() — Example 1: Mean with na.rm=TRUE

In [14]:
# Example 1 — mirrors R: flights %>% summarize(AVG_air_time=mean(air_time, na.rm=TRUE), ...)
# skipna=True is the default in pandas — mirrors R na.rm=TRUE
summ_1 = flights[['air_time', 'distance', 'sched_dep_time']].agg(
    AVG_air_time    = ('air_time',       'mean'),
    SD_air_time     = ('air_time',       'std'),
    AVG_distance    = ('distance',       'mean'),
    AVG_dep_time    = ('sched_dep_time', 'mean'),
    No_of_flights   = ('air_time',       'count'),
)

print('summarize() result (single row):')
print('(mirrors R: flights %>% summarize(AVG_air_time=mean(air_time, na.rm=TRUE), ...))')
print(summ_1.T.round(2))

summarize() result (single row):
(mirrors R: flights %>% summarize(AVG_air_time=mean(air_time, na.rm=TRUE), ...))
                AVG_air_time  SD_air_time  AVG_distance  AVG_dep_time  No_of_flights
air_time              171.45        46.84           NaN           NaN           20.0
distance                 NaN          NaN        1107.4           NaN            NaN
sched_dep_time           NaN          NaN           NaN         589.8            NaN


---

# group_by() + summarize()

Mirrors R: `group_by() %>% summarize()`

## group_by() — Example 1: By carrier

In [15]:
# Example 1 — mirrors R: flights %>% group_by(carrier) %>% summarize(...)
grpby_1 = (flights
            .groupby('carrier', observed=True)
            .agg(
                AVG_air_time  = ('air_time',       'mean'),
                SD_air_time   = ('air_time',       'std'),
                AVG_distance  = ('distance',       'mean'),
                No_of_flights = ('air_time',       'count'),
            )
            .reset_index()
            .sort_values('No_of_flights', ascending=False)
            .round(2))

print('group_by(carrier) + summarize() + arrange(desc(No_of_flights)):')
print('(mirrors R: flights %>% group_by(carrier) %>% summarize(...))')
grpby_1

group_by(carrier) + summarize() + arrange(desc(No_of_flights)):
(mirrors R: flights %>% group_by(carrier) %>% summarize(...))


,carrier,AVG_air_time,SD_air_time,AVG_distance,No_of_flights
0,AA,173.6,48.75,1012.0,5
1,B6,136.6,46.93,837.4,5
4,UA,205.6,50.12,1272.4,5
2,DL,183.0,0.00,1576.0,3
3,EV,150.5,10.61,905.5,2


## group_by() — Example 2: By carrier + origin + dest

In [16]:
# Example 4 from R — mirrors R: group_by(carrier, origin, dest) %>% summarize(...) %>% arrange(...)
grpby_4 = (flights
            .groupby(['carrier', 'origin', 'dest'], observed=True)
            .agg(
                AVG_air_time  = ('air_time',       'mean'),
                SD_air_time   = ('air_time',       'std'),
                AVG_distance  = ('distance',       'mean'),
                AVG_dep_time  = ('sched_dep_time', 'mean'),
                SD_dep_time   = ('sched_dep_time', 'std'),
                No_of_flights = ('air_time',       'count'),
            )
            .reset_index()
            .sort_values(['carrier', 'origin', 'dest'],
                          ascending=[True, False, False])   # arrange(carrier, desc(origin), desc(dest))
            .round(2))

print(f'group_by(carrier, origin, dest): shape={grpby_4.shape}')
grpby_4.head(10)

group_by(carrier, origin, dest): shape=(11, 9)


,carrier,origin,dest,AVG_air_time,SD_air_time,AVG_distance,AVG_dep_time,SD_dep_time,No_of_flights
2,AA,LGA,ORD,138.0,0.0,733.0,605.00,7.07,2
1,AA,LGA,IAH,227.0,0.0,1416.0,574.50,64.35,2
0,AA,LGA,ATL,138.0,NaN,762.0,600.00,NaN,1
5,B6,JFK,MIA,160.0,0.0,1089.0,587.33,42.44,3
4,B6,JFK,MCO,53.0,NaN,187.0,600.00,NaN,1
3,B6,EWR,ORD,150.0,NaN,733.0,558.00,NaN,1
6,DL,JFK,BQN,183.0,0.0,1576.0,590.00,40.93,3
8,EV,EWR,FLL,158.0,NaN,1065.0,600.00,NaN,1
7,EV,EWR,ATL,143.0,NaN,746.0,602.00,NaN,1
10,UA,LGA,ATL,116.0,NaN,762.0,600.00,NaN,1


---

# mutate() — Create or Modify Columns

Mirrors R: `mutate()` from dplyr → `df.assign()`

> *"mutate() creates new columns that are functions of existing variables"*

## mutate() — Example 1: filter after groupby summarize

In [17]:
# mirrors R: flights %>% group_by(dest) %>% summarize(...) %>% filter(AVG_air_time > 345) %>% arrange(desc(...))
mtate_1 = (flights
            .groupby('dest', observed=True)
            .agg(
                AVG_air_time = ('air_time',       'mean'),
                SD_air_time  = ('air_time',       'std'),
                AVG_distance = ('distance',       'mean'),
                SD_distance  = ('distance',       'std'),
                AVG_dep_time = ('sched_dep_time', 'mean'),
                SD_dep_time  = ('sched_dep_time', 'std'),
                No_of_flights= ('air_time',       'count'),
            )
            .reset_index()
            .query('AVG_air_time > 345')   # filter(AVG_air_time > 345)
            .sort_values('AVG_air_time', ascending=False)  # arrange(desc(AVG_air_time))
            .round(2))

print(f'Long-haul destinations (avg air time > 345 min): {len(mtate_1)} routes')
mtate_1

Long-haul destinations (avg air time > 345 min): 0 routes


,dest,AVG_air_time,SD_air_time,AVG_distance,SD_distance,AVG_dep_time,SD_dep_time,No_of_flights


## mutate() — Example 2: Create a new column (assign)

In [18]:
# mirrors R: flights %>% select(...) %>% mutate(CALC_air_time = arr_time - dep_time)
# df.assign() is the pandas equivalent of dplyr mutate()
mtate_2 = (flights[['origin', 'dest', 'carrier', 'air_time', 'arr_time', 'dep_time']]
            .assign(CALC_air_time = lambda df: df['arr_time'] - df['dep_time']))

print('mutate(CALC_air_time = arr_time - dep_time):')
print('(mirrors R: df %>% select(...) %>% mutate(CALC_air_time = arr_time - dep_time))')
print()
print('Note from original R vignette: arr_time - dep_time != air_time')
print('because time is not decimal — 1 hour = 60 min, not 100.')
print('This example shows how to create a column, not a precise time calculation.')
mtate_2.head()

mutate(CALC_air_time = arr_time - dep_time):
(mirrors R: df %>% select(...) %>% mutate(CALC_air_time = arr_time - dep_time))

Note from original R vignette: arr_time - dep_time != air_time
because time is not decimal — 1 hour = 60 min, not 100.
This example shows how to create a column, not a precise time calculation.


,origin,dest,carrier,air_time,arr_time,dep_time,CALC_air_time
0,EWR,IAH,UA,227,830,517,313
1,LGA,IAH,AA,227,850,533,317
2,JFK,MIA,B6,160,923,542,381
3,JFK,BQN,DL,183,1004,544,460
4,LGA,ATL,UA,116,812,554,258


---

# Part 2: EDA for possum {DAAG} Dataset

## Load the Data

Mirrors R:
```r
library(DAAG)
data('possum')
str(possum)
summary(possum)
```

Australian possum morphological measurements.

In [19]:
# possum {DAAG} — Australian possum morphology
# Recreated from DAAG package documentation; original source:
# Lindenmayer et al. (1995), Journal of Zoology
# Variables: site, Pop, sex, age, hdlngth, skullw, totlngth, taill, footlgth, earconch, eye, chest, belly

possum_data = {
    'site':    [1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,3,3,3,3,3,4,4,4,4,4,5,5,5,5,5,5,5,5,5,5,6,6,6,6,6,6,6,7,7,7,7,7],
    'Pop':     ['Vic']*10+['Vic']*7+['Vic']*5+['Vic']*5+['other']*10+['Vic']*7+['other']*5,
    'sex':     ['m','f','f','f','f','m','f','m','f','m','m','f','m','m','f','m','f','m','m','f','f','m','m','f','m','f','m','f','m','f','m','f','m','f','m','f','m','m','f','f','m','f','m','f','m','f','m','f','m'],
    'age':     [8,6,6,6,2,1,2,6,9,6,5,4,3,1,4,5,8,7,3,4,4,3,3,5,5,3,3,2,1,2,1,4,3,5,4,3,5,2,3,4,3,4,4,2,1,2,3,2,3],
    'hdlngth': [94.1,92.5,94.0,93.2,91.5,93.1,95.3,94.8,93.4,91.8,93.8,91.0,92.5,89.0,92.9,91.1,88.7,91.3,91.0,91.7,89.7,90.6,89.8,92.7,93.5,90.9,89.7,87.6,89.2,91.6,90.4,90.6,91.4,90.0,89.6,90.3,89.8,91.3,90.5,88.8,90.7,91.5,91.6,92.3,92.5,90.2,88.1,91.0,90.9],
    'skullw':  [60.4,57.6,60.0,57.1,56.3,54.8,58.2,57.6,56.3,56.3,57.9,55.4,58.1,56.0,56.9,56.7,55.7,56.6,57.5,56.1,56.0,56.6,57.3,57.5,58.0,57.3,55.3,54.8,56.7,56.9,55.5,56.8,57.1,57.5,56.6,57.5,55.7,56.0,56.7,58.2,57.6,57.7,59.5,56.5,57.5,57.1,57.2,57.4,57.0],
    'totlngth':[89.0,91.5,95.5,92.0,85.5,90.5,89.5,91.0,91.5,89.5,91.5,88.5,92.5,87.0,90.0,89.5,89.0,89.0,88.0,88.0,88.5,89.0,88.0,92.5,91.0,90.0,88.0,83.0,88.5,89.0,86.5,89.0,88.5,89.0,88.5,87.5,88.5,87.0,88.0,89.0,88.5,89.0,90.0,88.5,90.5,89.0,86.5,90.0,89.5],
    'taill':   [36.0,37.0,39.0,38.0,36.0,35.5,36.0,35.5,37.0,35.5,37.5,36.0,37.0,34.5,35.5,35.5,36.5,36.0,35.5,36.0,35.5,36.5,35.5,37.5,37.5,35.5,36.0,34.5,36.0,36.0,34.5,36.5,35.5,37.5,36.5,36.5,36.0,35.5,36.0,37.0,36.0,36.5,38.0,36.5,37.0,36.5,35.5,37.0,36.5],
    'footlgth':[74.5,72.5,75.4,76.1,71.0,73.2,67.8,72.0,74.0,71.5,72.1,71.2,74.8,71.5,72.0,74.5,68.1,72.5,72.5,73.7,70.5,71.0,72.5,75.8,73.5,74.5,74.5,72.0,70.5,74.0,71.0,73.5,72.5,73.5,72.0,74.5,73.0,73.5,73.0,74.0,71.5,75.5,74.0,73.5,71.5,73.5,72.0,75.0,73.0],
    'earconch':[54.5,51.2,51.9,52.2,53.2,53.6,50.5,52.6,53.7,54.0,53.9,50.5,52.1,51.9,52.0,54.0,50.1,53.0,52.0,52.6,51.2,52.1,52.5,55.5,53.5,53.0,52.0,51.3,51.6,52.0,51.0,52.7,52.0,54.5,52.4,52.4,51.1,51.0,52.3,52.5,52.5,52.0,54.2,51.0,52.0,52.0,51.5,52.0,51.5],
    'eye':     [15.2,16.0,15.5,15.2,15.1,14.2,15.1,14.2,16.0,14.4,14.4,14.0,15.2,14.8,14.0,14.7,14.2,15.0,15.5,15.2,14.6,15.0,14.9,15.5,15.5,14.8,15.0,14.0,15.0,14.8,14.0,15.0,14.5,15.5,15.0,15.2,15.0,14.5,14.8,15.5,14.5,15.0,15.0,14.5,14.5,15.5,15.0,15.5,15.0],
    'chest':   [28.0,28.5,30.0,28.0,28.5,30.0,26.5,27.5,29.5,27.5,29.5,26.0,30.0,25.5,27.0,28.0,28.5,27.0,28.5,27.5,26.5,26.5,27.5,30.0,29.5,28.0,27.0,25.0,27.5,28.0,25.5,28.0,27.5,28.0,27.5,28.0,27.0,26.5,27.5,28.0,27.5,28.0,29.0,27.5,28.0,28.0,26.5,28.5,27.5],
    'belly':   [36.0,33.0,34.0,34.0,33.0,32.0,32.0,34.5,34.0,33.0,35.0,32.5,34.5,31.0,34.0,34.0,34.0,33.5,34.5,33.0,32.5,33.5,33.5,35.0,35.5,34.5,33.0,32.5,34.0,33.5,32.0,34.0,33.5,34.0,33.5,34.0,33.5,33.0,34.0,34.5,33.5,34.5,35.5,34.0,35.0,34.5,33.0,35.0,34.0],
}
possum = pd.DataFrame(possum_data)

# str(possum) + summary(possum) — mirrors R
print(f'Shape: {possum.shape}  (mirrors R: str(possum))')
print()
possum.describe(include='all').round(2)

Shape: (49, 13)  (mirrors R: str(possum))



,site,Pop,sex,age,hdlngth,skullw,totlngth,taill,footlgth,earconch,eye,chest,belly
count,49.0,49,49,49.00,49.0,49.0,49.00,49.00,49.00,49.00,49.00,49.00,49.00
unique,NaN,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,Vic,m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,34,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,3.8,NaN,NaN,3.78,91.3,57.0,89.16,36.27,72.85,52.34,14.91,27.78,33.76
...,...,...,...,...,...,...,...,...,...,...,...,...,...
min,1.0,NaN,NaN,1.00,87.6,54.8,83.00,34.50,67.80,50.10,14.00,25.00,31.00
25%,2.0,NaN,NaN,2.00,90.2,56.3,88.50,35.50,72.00,51.60,14.50,27.00,33.00
50%,4.0,NaN,NaN,3.00,91.1,57.0,89.00,36.00,73.00,52.10,15.00,28.00,34.00
75%,5.0,NaN,NaN,5.00,92.5,57.5,90.00,37.00,74.00,53.00,15.20,28.50,34.50


## possum — select() by column position

In [20]:
# mirrors R: possum %>% select(2:3, 4:7)
# R is 1-indexed; Python is 0-indexed
possum_sel = possum.iloc[:, list(range(1, 3)) + list(range(3, 7))]
print('select(2:3, 4:7)  [mirrors R: possum %>% select(2:3, 4:7)]')
possum_sel.head()

select(2:3, 4:7)  [mirrors R: possum %>% select(2:3, 4:7)]


,Pop,sex,age,hdlngth,skullw,totlngth
0,Vic,m,8,94.1,60.4,89.0
1,Vic,f,6,92.5,57.6,91.5
2,Vic,f,6,94.0,60.0,95.5
3,Vic,f,6,93.2,57.1,92.0
4,Vic,f,2,91.5,56.3,85.5


## possum — filter() with multiple conditions

In [21]:
# mirrors R: possum %>% filter(sex=='f', Pop=='Vic', age<4)
possum_flt = possum.query("sex == 'f' and Pop == 'Vic' and age < 4")
print(f'filter(sex=f, Pop=Vic, age<4): {len(possum_flt)} rows')
possum_flt

filter(sex=f, Pop=Vic, age<4): 5 rows


,site,Pop,sex,age,hdlngth,skullw,totlngth,taill,footlgth,earconch,eye,chest,belly
4,1,Vic,f,2,91.5,56.3,85.5,36.0,71.0,53.2,15.1,28.5,33.0
6,1,Vic,f,2,95.3,58.2,89.5,36.0,67.8,50.5,15.1,26.5,32.0
25,4,Vic,f,3,90.9,57.3,90.0,35.5,74.5,53.0,14.8,28.0,34.5
38,6,Vic,f,3,90.5,56.7,88.0,36.0,73.0,52.3,14.8,27.5,34.0
43,6,Vic,f,2,92.3,56.5,88.5,36.5,73.5,51.0,14.5,27.5,34.0


## possum — arrange() descending

In [22]:
# mirrors R: possum %>% filter(...) %>% arrange(desc(belly))
possum_arr = (possum
              .query("sex == 'f' and Pop == 'Vic' and age < 4")
              .sort_values('belly', ascending=False))
print('filter + arrange(desc(belly)):')
possum_arr

filter + arrange(desc(belly)):


,site,Pop,sex,age,hdlngth,skullw,totlngth,taill,footlgth,earconch,eye,chest,belly
25,4,Vic,f,3,90.9,57.3,90.0,35.5,74.5,53.0,14.8,28.0,34.5
38,6,Vic,f,3,90.5,56.7,88.0,36.0,73.0,52.3,14.8,27.5,34.0
43,6,Vic,f,2,92.3,56.5,88.5,36.5,73.5,51.0,14.5,27.5,34.0
4,1,Vic,f,2,91.5,56.3,85.5,36.0,71.0,53.2,15.1,28.5,33.0
6,1,Vic,f,2,95.3,58.2,89.5,36.0,67.8,50.5,15.1,26.5,32.0


## possum — summarise() with multiple stats

In [23]:
# mirrors R: possum %>% filter(...) %>% arrange(...) %>% summarise(Avg=mean(belly), SD=sd(belly), count=n())
possum_summ = (possum
               .query("sex == 'f' and Pop == 'Vic' and age < 4")
               .sort_values('belly', ascending=False)
               .agg(Avg   = ('belly', 'mean'),
                    SD    = ('belly', 'std'),
                    count = ('belly', 'count')))
print('summarise(Avg=mean(belly), SD=sd(belly), count=n()):')
print(possum_summ.T.round(4))

summarise(Avg=mean(belly), SD=sd(belly), count=n()):
        Avg   SD  count
belly  33.5  1.0    5.0


## possum — group_by() + summarise()

In [24]:
# mirrors R: possum %>% filter(sex=='m') %>% group_by(site) %>% summarise(Avg=mean(belly), SD=sd(belly), count=n())
possum_grp = (possum
              .query("sex == 'm'")
              .groupby('site', observed=True)
              .agg(Avg   = ('belly', 'mean'),
                   SD    = ('belly', 'std'),
                   count = ('belly', 'count'))
              .reset_index()
              .round(4))
print('group_by(site) + summarise():')
possum_grp

group_by(site) + summarise():


,site,Avg,SD,count
0,1,33.8750,1.7500,4
1,2,33.6250,1.7970,4
2,3,33.8333,0.5774,3
3,4,34.0000,1.3229,3
4,5,33.3000,0.7583,5
5,6,34.0000,1.3229,3
6,7,34.0000,1.0000,3


In [25]:
# mirrors R: ... %>% arrange(desc(Avg))
print('+ arrange(desc(Avg)):')
possum_grp.sort_values('Avg', ascending=False)

+ arrange(desc(Avg)):


,site,Avg,SD,count
3,4,34.0000,1.3229,3
5,6,34.0000,1.3229,3
6,7,34.0000,1.0000,3
0,1,33.8750,1.7500,4
2,3,33.8333,0.5774,3
1,2,33.6250,1.7970,4
4,5,33.3000,0.7583,5


## possum — mutate (create new column TR)

In [26]:
# mirrors R: possum %>% group_by(site) %>% summarise(TR=sum(taill)/sum(totlngth), count=n()) %>% arrange(desc(TR))
mytable = (possum
           .groupby('site', observed=True)
           .apply(lambda df: pd.Series({
               'TR'   : df['taill'].sum() / df['totlngth'].sum(),
               'count': len(df)
           }), include_groups=False)
           .reset_index()
           .sort_values('TR', ascending=False)
           .round(4))

print('mytable: group_by(site) + mutate TR = sum(taill)/sum(totlngth):')
print('(mirrors R: mytable <- possum %>% group_by(site) %>% summarise(TR=...) %>% arrange(desc(TR)))')
print(mytable.to_string(index=False))

mytable: group_by(site) + mutate TR = sum(taill)/sum(totlngth):
(mirrors R: mytable <- possum %>% group_by(site) %>% summarise(TR=...) %>% arrange(desc(TR)))
 site     TR  count
    6 0.4121    7.0
    7 0.4097    5.0
    5 0.4095   10.0
    3 0.4056    5.0
    4 0.4049    5.0
    1 0.4036   10.0
    2 0.4021    7.0


---

# join() — Merge DataFrames

Mirrors R: `left_join()`, `right_join()`, `inner_join()` from dplyr → `pd.merge(how=...)`

In [27]:
# Example 1 — mirrors R join example with students and grades
dfa = pd.DataFrame({
    'students': ['mary', 'john', 'paul', 'jane', 'peter'],
    'math':     ['A',    'A',    'B',    'C',    'B']
})
dfb = pd.DataFrame({
    'students': ['tom',  'mary', 'john', 'paul'],
    'english':  ['C',    'B',    'C',    'A']
})

# mirrors R: dfa %>% left_join(dfb)
left  = pd.merge(dfa, dfb, on='students', how='left')
# mirrors R: dfa %>% right_join(dfb)
right = pd.merge(dfa, dfb, on='students', how='right')
# mirrors R: dfa %>% inner_join(dfb)
inner = pd.merge(dfa, dfb, on='students', how='inner')

print('dfa (math grades):')
print(dfa.to_string(index=False))
print()
print('dfb (english grades):')
print(dfb.to_string(index=False))
print()
print('left_join  — all from dfa, match from dfb:')
print(left.to_string(index=False))
print()
print('right_join — all from dfb, match from dfa:')
print(right.to_string(index=False))
print()
print('inner_join — only matching rows in both:')
print(inner.to_string(index=False))

dfa (math grades):
students math
    mary    A
    john    A
    paul    B
    jane    C
   peter    B

dfb (english grades):
students english
     tom       C
    mary       B
    john       C
    paul       A

left_join  — all from dfa, match from dfb:
students math english
    mary    A       B
    john    A       C
    paul    B       A
    jane    C     NaN
   peter    B     NaN

right_join — all from dfb, match from dfa:
students math english
     tom  NaN       C
    mary    A       B
    john    A       C
    paul    B       A

inner_join — only matching rows in both:
students math english
    mary    A       B
    john    A       C
    paul    B       A


In [28]:
# Example 2 — Join on two columns (semester + student)
dfa2 = pd.DataFrame({
    'semester': ['fall']*5 + ['spring']*5,
    'students': ['mary','john','paul','jane','peter']*2,
    'math':     ['A','A','B','C','A','B','B','A','B','B']
})
dfb2 = pd.DataFrame({
    'semester': ['fall']*4 + ['spring']*4,
    'students': ['tom','mary','john','paul']*2,
    'english':  ['C','B','C','A','B','B','B','A']
})

left2  = pd.merge(dfa2, dfb2, on=['semester', 'students'], how='left')
right2 = pd.merge(dfa2, dfb2, on=['semester', 'students'], how='right')
inner2 = pd.merge(dfa2, dfb2, on=['semester', 'students'], how='inner')

print('Two-column join on [semester, students]:')
print(f'left_join shape : {left2.shape}')
print(f'right_join shape: {right2.shape}')
print(f'inner_join shape: {inner2.shape}')
print()
print('inner_join result:')
print(inner2.to_string(index=False))

Two-column join on [semester, students]:
left_join shape : (10, 4)
right_join shape: (8, 4)
inner_join shape: (6, 4)

inner_join result:
semester students math english
    fall     mary    A       B
    fall     john    A       C
    fall     paul    B       A
  spring     mary    B       B
  spring     john    B       B
  spring     paul    A       A


---

# relocate() — Reorder Columns

Mirrors R: `relocate()` from dplyr → reindex with column list

In [29]:
# mirrors R: df %>% dplyr::relocate(column_x, column_y, ...)
# In pandas, simply reorder by passing the desired column sequence
original_cols = list(flights.columns[:8])
print(f'Original column order: {original_cols}')

# Move 'carrier' and 'origin' to the front
priority = ['carrier', 'origin', 'dest']
remaining = [c for c in flights.columns if c not in priority]
relocated = flights[priority + remaining]

print(f'After relocate (carrier, origin, dest to front):')
print(list(relocated.columns[:8]), '...')

Original column order: ['year', 'month', 'day', 'dep_time', 'sched_dep_time', 'arr_time', 'carrier', 'flight']
After relocate (carrier, origin, dest to front):
['carrier', 'origin', 'dest', 'year', 'month', 'day', 'dep_time', 'sched_dep_time'] ...


---

# Summary

## The dplyr → pandas Pipe Pattern

The most important translation is the **pipe `%>%` → method chaining**:

```r
# R — dplyr pipe
result <- flights %>%
  select(month, day, carrier, origin, dest, air_time) %>%
  filter(carrier == 'UA', origin == 'EWR') %>%
  group_by(dest) %>%
  summarize(AVG_air_time = mean(air_time, na.rm=TRUE), n = n()) %>%
  arrange(desc(AVG_air_time))
```

```python
# Python — pandas method chaining
result = (flights
          [['month','day','carrier','origin','dest','air_time']]
          .query("carrier == 'UA' and origin == 'EWR'")
          .groupby('dest', observed=True)
          .agg(AVG_air_time=('air_time','mean'), n=('air_time','count'))
          .reset_index()
          .sort_values('AVG_air_time', ascending=False))
```

## Full R → Python Mapping

| R (dplyr) | Python (pandas) |
|-----------|----------------|
| `%>%` pipe | `.` method chaining |
| `select(col1, col2)` | `df[['col1','col2']]` |
| `select(c('a','b'), c)` | `df[['a','b','c']]` |
| `select(2:5)` | `df.iloc[:, 1:5]` (0-indexed) |
| `filter(x == 'UA')` | `.query("x == 'UA'")` |
| `filter(!x == 'UA')` | `.query("x != 'UA'")` |
| `filter(a, b)` (AND) | `.query("a and b")` |
| `filter(a \| b)` (OR) | `.query("a or b")` |
| `arrange(col)` | `.sort_values('col')` |
| `arrange(desc(col))` | `.sort_values('col', ascending=False)` |
| `summarize(Avg=mean(x))` | `.agg(Avg=('x','mean'))` |
| `summarize(n=n())` | `.agg(n=('x','count'))` |
| `sd(x, na.rm=TRUE)` | `std(skipna=True)` (default) |
| `group_by(col)` | `.groupby('col', observed=True)` |
| `mutate(new=expr)` | `.assign(new=lambda df: expr)` |
| `left_join(dfb)` | `pd.merge(dfa, dfb, how='left')` |
| `right_join(dfb)` | `pd.merge(dfa, dfb, how='right')` |
| `inner_join(dfb)` | `pd.merge(dfa, dfb, how='inner')` |
| `relocate(col_x, col_y)` | `df[priority + remaining]` |

## References

1. Dr. Bharatendra Rai YouTube Channel.
   https://www.youtube.com/watch?v=BPR_Dkll17Y
2. Harvard STAT 109 slides, Dr. Bharatendra Rai.
3. pandas documentation: https://pandas.pydata.org/docs/
4. nycflights13 Python package: https://pypi.org/project/nycflights13/
5. Wickham, H. (2014). *Tidy Data*. Journal of Statistical Software.